# ISLR TFRecords by signer fold (Kaggle, CPU)

Converts the `asl-signs` competition parquets into the record format of the 1st-place solution
(`coordinates`: float32 (T, 543, 3) raw MediaPipe x/y/z with NaNs kept, `sign`: int64), GZIP, sharded
as `fold{k}-{shard}-{count}.tfrecords` so its `train_folds(CFG, [k])` holds out exactly our signer fold
k (`--val-fold k` in this repo). Each record also carries `participant_id` and `sequence_id` (ignored by
the 1st-place parser; used for per-signer evaluation and ensembling).

Add the **asl-signs** competition as input. Writes to the disk with the most free space and publishes
`DATASET_ID` (new version, or created).

In [ ]:
!pip install -q lmdb >/dev/null 2>&1 || true

In [ ]:
import json, os, shutil, subprocess, sys, time
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import numpy as np, pandas as pd
import tensorflow as tf
from tqdm import tqdm

In [ ]:
COMP_DIR = Path("/kaggle/input/competitions/asl-signs")
if not COMP_DIR.exists():
    COMP_DIR = Path("/kaggle/input/asl-signs")
DATASET_ID = "shravnchandr/islr-tfrecords-signer-folds"
DATASET_TITLE = "ISLR TFRecords by signer fold"
SHARD_SIZE = 2000
NUM_WORKERS = os.cpu_count() or 4
UPLOAD = True
ROWS_PER_FRAME = 543

# GroupKFold(7) by participant_id, as cnn_transformer --val-fold (hard-coded, sklearn-version proof)
FOLDS = {"0": [34503, 49445, 62590], "1": [27610, 37055, 61333], "2": [25571, 36257, 53618], "3": [16069, 22343, 28656], "4": [18796, 26734, 29302], "5": [32319, 4718, 55372], "6": [2044, 30680, 37779]}
FOLD_OF = {int(p): int(k) for k, ps in FOLDS.items() for p in ps}

free = {}
for c in (Path("/kaggle/tmp"), Path("/tmp"), Path("/kaggle/working")):
    try:
        c.mkdir(parents=True, exist_ok=True); free[c] = shutil.disk_usage(c).free / 1024**3
    except OSError:
        pass
for c, gb in free.items():
    print(f"  {str(c):18s} {gb:6.1f} GB free")
DS_DIR = max(free, key=free.get) / "islr-tfrecords"
DS_DIR.mkdir(parents=True, exist_ok=True)
print("Building in", DS_DIR)

In [ ]:
def load_coords(path):
    # The competition's reference loader: 543 rows per frame, frame-major, x/y/z, NaN = not detected.
    data = pd.read_parquet(path, columns=["x", "y", "z"])
    return data.values.reshape(len(data) // ROWS_PER_FRAME, ROWS_PER_FRAME, 3).astype(np.float32)


def _feature(v, kind):
    if kind == "bytes":
        return tf.train.Feature(bytes_list=tf.train.BytesList(value=[v]))
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[int(v)]))


def write_shard(args):
    rows, out_path = args
    n = 0
    with tf.io.TFRecordWriter(str(out_path) + ".part", options="GZIP") as w:
        for path, sign, pid, sid in rows:
            x = load_coords(COMP_DIR / path)
            ex = tf.train.Example(features=tf.train.Features(feature={
                "coordinates": _feature(x.tobytes(), "bytes"),
                "sign": _feature(sign, "int"),
                "participant_id": _feature(pid, "int"),
                "sequence_id": _feature(sid, "int"),
            }))
            w.write(ex.SerializeToString()); n += 1
    final = Path(str(out_path).replace("{count}", str(n)))
    os.rename(str(out_path) + ".part", final)
    return final.name, n

In [ ]:
df = pd.read_csv(COMP_DIR / "train.csv")
sign2idx = json.load(open(COMP_DIR / "sign_to_prediction_index_map.json"))
df["label"] = df["sign"].map(sign2idx)
df["fold"] = df["participant_id"].map(FOLD_OF)
assert df["fold"].notna().all(), "signer without a fold"
print(df.groupby("fold").agg(n=("path", "size"), signers=("participant_id", lambda s: sorted(s.unique()))))

tasks = []
for k, g in df.groupby("fold"):
    g = g.sample(frac=1.0, random_state=42)  # shuffle within the fold before sharding
    for i in range(0, len(g), SHARD_SIZE):
        part = g.iloc[i:i + SHARD_SIZE]
        rows = list(zip(part["path"], part["label"], part["participant_id"], part["sequence_id"]))
        tasks.append((rows, DS_DIR / f"fold{int(k)}-{i // SHARD_SIZE:02d}-{{count}}.tfrecords"))
print(f"{len(tasks)} shards, {len(df):,} samples")

t0, written = time.time(), 0
# fork: workers inherit the notebook-defined functions (spawn/forkserver cannot import __main__)
with ProcessPoolExecutor(NUM_WORKERS, mp_context=mp.get_context("fork")) as ex:
    for name, n in tqdm(ex.map(write_shard, tasks), total=len(tasks), mininterval=30):
        written += n
print(f"Done in {(time.time() - t0) / 60:.0f} min: {written:,} records")
assert written == len(df)
df[["path", "participant_id", "sequence_id", "sign", "fold"]].to_csv(DS_DIR / "train.csv", index=False)
shutil.copy(COMP_DIR / "sign_to_prediction_index_map.json", DS_DIR)
print(f"Size: {sum(f.stat().st_size for f in DS_DIR.glob('*.tfrecords')) / 1024**3:.1f} GiB")

In [ ]:
# Verify: every shard parses in the 1st-place format and the per-file counts add up.
import re
files = sorted(DS_DIR.glob("*.tfrecords"))
counts = [int(re.compile(r"-([0-9]*)\.").search(f.name).group(1)) for f in files]
assert sum(counts) == len(df), (sum(counts), len(df))
ds = tf.data.TFRecordDataset([str(files[0])], compression_type="GZIP")
for rec in ds.take(3):
    f = tf.io.parse_single_example(rec, {"coordinates": tf.io.FixedLenFeature([], tf.string),
                                         "sign": tf.io.FixedLenFeature([], tf.int64)})
    x = tf.reshape(tf.io.decode_raw(f["coordinates"], tf.float32), (-1, ROWS_PER_FRAME, 3))
    print(x.shape, int(f["sign"]))
print({k: sum(c for f, c in zip(files, counts) if f.name.startswith(f"fold{k}-")) for k in range(7)})

In [ ]:
if UPLOAD:
    (DS_DIR / "dataset-metadata.json").write_text(json.dumps(
        {"title": DATASET_TITLE, "id": DATASET_ID, "licenses": [{"name": "other"}]}))
    r = subprocess.run(["kaggle", "datasets", "version", "-p", str(DS_DIR), "-m", "signer-fold TFRecords"],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        r = subprocess.run(["kaggle", "datasets", "create", "-p", str(DS_DIR)], capture_output=True, text=True)
        print(r.stdout or r.stderr)
    r.check_returncode()